In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp1d
from scipy.ndimage import label
import optuna
from collections import deque
from scripts.loader import * 
from scripts.processing import *
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import LeaveOneOut

# All participants

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

folder_lopo = Path("./LOPO_seed0/")
folder_loso = Path("./LOSO_seed100/")

speeds = [1.0, 1.2, 1.4, 1.6, 1.8]  # adapt if needed
participants = ['S03', 'S04', 'S05', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12',
       'S13', 'S14', 'S15', 'S16', 'S17', 'S18']

for p in participants:
    fig, axes = plt.subplots(
        1, len(speeds),
        figsize=(4 * len(speeds), 3),
        sharey=False
    )

    for ax, speed in zip(axes, speeds):
        filename = f"{p}_speed{speed}_predicted_speed.csv"

        csv_loso = folder_loso / filename
        csv_lopo = folder_lopo / filename

        if not csv_loso.exists() or not csv_lopo.exists():
            ax.set_title(f"{speed} m/s\nmissing")
            ax.axis("off")
            continue

        df_loso = pd.read_csv(csv_loso)
        df_lopo = pd.read_csv(csv_lopo)

        ax.plot(df_loso["measured_speed"], label="measured", linewidth=2)
        ax.plot(df_loso["predicted_speed"], label="seed 100")
        ax.plot(df_lopo["predicted_speed"], label="seed 0")

        ax.set_title(f"{speed} m/s")
        ax.set_xlabel("Sample")

    axes[0].set_ylabel("Speed")
    axes[-1].legend()

    fig.suptitle(f"Participant {p}", y=1.05)
    fig.tight_layout()
    plt.show()

## selected participants

In [ ]:
def shade_regions(ax, mask, color="lightgray", alpha=0.3):
    mask = np.asarray(mask)

    starts = np.where(np.diff(mask.astype(int)) == 1)[0] + 1
    ends = np.where(np.diff(mask.astype(int)) == -1)[0] + 1

    if mask[0]:
        starts = np.r_[0, starts]
    if mask[-1]:
        ends = np.r_[ends, len(mask)]

    for start, end in zip(starts, ends):
        ax.axvspan(start, end, color=color, alpha=alpha, zorder=0)

In [ ]:
x = np.arange(len(df_loso))

swing = np.isclose(df_loso["predicted_speed"], speed, atol=1e-6)
swing = swing & (x > len(df_loso) * 0.5)

shade_regions(ax, swing)

In [ ]:
palette = sns.color_palette("rocket_r", 10)

color_measured = palette[0]       # dark purple
color_unseen_speed = "#D51C3C"   # reddish/pink
color_unseen_participant = "#F69F19"  # light orange

swing_color  = "#F0F1F2"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
import numpy as np

participants = ["S03", "S12", "S15", "S17"]   # 4 participants

ps = {"S03":"Participant 1", "S12": "Participant 9", "S15": "Participant 12", "S17": "Participant 14"}
speeds = [1.0, 1.4, 1.8]                      # 3 speeds

fig, axes = plt.subplots(
    len(speeds), len(participants),
    figsize=(11, 6.5),
    sharex=True,
    sharey='row'
)

for i, speed in enumerate(speeds):
    for j, participant in enumerate(participants):

        
      

        ax = axes[i, j]

        filename = f"{participant}_speed{speed}_predicted_speed.csv"

        df_loso = pd.read_csv(folder_loso / filename)
        df_lopo = pd.read_csv(folder_lopo / filename)

        x = np.arange(len(df_loso))
        swing = np.isclose(df_loso["predicted_speed"], speed, atol=1e-6)
        swing = swing & (x > len(df_loso) * 0.5)

        shade_regions(ax, swing, color="#F0F0F0", alpha = 1)



        ax.plot(df_loso["measured_speed"],
                color="gray",
                linewidth=2,
                label="measured"
               )

        ax.plot(df_loso["predicted_speed"],
                color=color_unseen_speed,
                linestyle="-",
                label="predicted, unseen speed")

        ax.axhline(
        speed,
        color="0.5",
        linestyle="--",
        linewidth=1,
        zorder=1, label = "nominal speed"
    )

        ax.plot(df_lopo["predicted_speed"],
                color=color_unseen_participant,
                linestyle="--",
                label="predicted, unseen participant")




     #   ax.hlines(speed, 0, len(df_loso)-1,
               #   colors="gray",
              #    linestyles="dashed",
              #    label="desired speed")



        sns.despine(ax=ax)

        # Column titles (participants)
        if i == 0:
            ax.set_title(ps[participant], fontsize=12)

        # Row labels (speeds)
        if j == 0:
            ax.set_ylabel("belt speed [m/s]", fontsize=12)
        else:
            ax.set_ylabel("")

        # Only bottom row gets x-labels
        if i == len(speeds) - 1:
            ax.set_xlabel("% of stride", fontsize=12)

        
        if speed == 1.8:
            ax.set_yticks([1.74, 1.76, 1.78, 1.8, 1.82])

handles, labels = axes[0, 0].get_legend_handles_labels()

# Add shaded region to legend
handles.append(Patch(facecolor="#F0F0F0" , alpha=1, label="swing"))
labels.append("swing phase")


# Reorder
order = [
    labels.index("measured"),
    labels.index("nominal speed"),
    labels.index("predicted, unseen speed"),
    labels.index("swing phase"),
    labels.index("predicted, unseen participant"),
]

handles = [handles[i] for i in order]
labels = [labels[i] for i in order]

fig.legend(handles, labels,
           loc="lower center",
           ncol=3,
           bbox_to_anchor=(0.5, -0.03),
           fontsize=12)


plt.tight_layout(rect=[0, 0.05, 1, 1])

fig.canvas.draw()

for i, speed in enumerate(speeds):
    for j in range(len(participants)):
        ax = axes[i, j]

        for tick, value in zip(ax.get_yticklabels(), ax.get_yticks()):
            if np.isclose(value, speed):
                tick.set_fontweight("bold")


plt.savefig("./plots_paper/loso.pdf",bbox_inches="tight" )